In [1]:
import tensorflow as tf
import numpy as np

2025-07-02 10:46:40.324186: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 10:46:40.332707: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 10:46:40.386415: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 10:46:40.441547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751453200.483529   17912 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751453200.49

In [2]:
from migration.datasets import create_AIS_dataset
inputs, targets, _, _, _, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   32,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)

Instructions for updating:
Use output_signature instead
Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    


2025-07-02 10:46:46.346206: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2025-07-02 10:46:46.733302: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [8]:
latent_size = 64
_DEFAULT_INITIALIZERS = {"w": tf.compat.v1.keras.initializers.VarianceScaling(scale=1.0, mode="fan_avg", distribution="uniform",seed=111),
                         "b": tf.compat.v1.zeros_initializer()}

_DEFAULT_INITIALIZERS_2 = {"w": tf.keras.initializers.VarianceScaling(scale=1.0, mode="fan_avg", distribution="uniform",seed=111),
                         "b": tf.zeros_initializer()}

weights sampled from [-limit, limit], where limit = sqrt(3 * scale / n) and n = (n_units_in + n_units_out)/2

Initializers: expect shapes

In [6]:
_DEFAULT_INITIALIZERS['w'](shape=(2,3))

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[ 0.92007613, -0.7151174 ,  0.17156708],
       [ 0.10697258,  0.6923902 , -0.25670475]], dtype=float32)>

In [9]:
_DEFAULT_INITIALIZERS_2['w'](shape=(2,3))

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[ 0.92007613, -0.7151174 ,  0.17156708],
       [ 0.10697258,  0.6923902 , -0.25670475]], dtype=float32)>

In [7]:
_DEFAULT_INITIALIZERS['b'](shape=(2,3))

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float32)>

In [10]:
_DEFAULT_INITIALIZERS_2['b'](shape=(2,3))

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float32)>

In [12]:
from migration.models.legacy.mlp import get_mlp

In [13]:
data_feat_extractor = get_mlp(
    layer_sizes=[latent_size, latent_size],
    initializers=_DEFAULT_INITIALIZERS_2,    
) 

In [14]:
data_feat_extractor

<Sequential name=sequential, built=False>

In [15]:
inputs_encoded = data_feat_extractor(inputs[1])

In [18]:
data_feat_extractor.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (32, 64)               │        44,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 64)               │         4,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,152 (192.00 KB)

 Trainable params: 49,152 (192.00 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
inputs[1]

<tf.Tensor: shape=(32, 702), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

In [20]:
inputs_encoded

<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
array([[0.0629536 , 0.        , 0.        , ..., 0.        , 0.00066966,
        0.05992217],
       [0.04518051, 0.        , 0.        , ..., 0.        , 0.04858114,
        0.12963076],
       [0.        , 0.        , 0.0804159 , ..., 0.03631388, 0.        ,
        0.00527748],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.14102437],
       [0.03455956, 0.        , 0.        , ..., 0.00526583, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.07055895, 0.01631805,
        0.0663643 ]], dtype=float32)>